In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
%run "../00.common/02.bronze-helpers.py"

In [0]:
# COMMAND ----------
 
# Load environment configuration
 
# MAGIC %run "../00_common/01_environment_config"
 
 
# COMMAND ----------
 
# Load Bronze helper functions
 
# MAGIC %run "../00_common/02_bronze_helpers"
 
 
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
source_file = f"{s3_root_path}/landing/drivers.json"
 
target_path = f"{s3_root_path}/Bronze/Nisarga/drivers"
 
print("Source File :", source_file)
print("Target Path :", target_path)
 
 
# COMMAND ----------
 
# ============================================================
# DEFINE SCHEMA
# ============================================================
 
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType
)
 
# Nested schema for driver name
name_schema = StructType([
    StructField("givenName", StringType(), True),
    StructField("familyName", StringType(), True)
])
 
# Driver schema
drivers_schema = StructType([
    StructField("driverId", StringType(), True),
    StructField("name", name_schema, True),
    StructField("dateOfBirth", DateType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])
 
 
# COMMAND ----------
 
# ============================================================
# READ DRIVERS DATA FROM S3 LANDING
# ============================================================
 
drivers_df = (
    spark.read
    .format("json")
    .schema(drivers_schema)
    .load(source_file)
)
 
display(drivers_df)
 
 
# COMMAND ----------
 
# Check schema
 
drivers_df.printSchema()
 
 
# COMMAND ----------
 
# Check source record count
 
print("Source Drivers Records:", drivers_df.count())
 
 
# COMMAND ----------
 
# ============================================================
# ADD INGESTION METADATA
# ============================================================
 
drivers_final_df = add_ingestion_metadata(drivers_df)
 
display(drivers_final_df)
 
 
# COMMAND ----------
 
# ============================================================
# WRITE DATA TO S3 BRONZE IN DELTA FORMAT
# ============================================================
 
(
    drivers_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(target_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# READ AND VALIDATE BRONZE DELTA DATA
# ============================================================
 
bronze_drivers_df = (
    spark.read
    .format("delta")
    .load(target_path)
)
 
display(bronze_drivers_df)
 
 
# COMMAND ----------
 
# Validate record count
 
print("Total Bronze Records:", bronze_drivers_df.count())
 
print("Bronze Drivers data successfully written to:")
print(target_path)